In [ ]:
# task5_tools.py - 투자 판단 분석 도구
from utils import *
from datetime import datetime, timedelta
import pandas as pd
import yfinance as yf
import requests
from typing import Dict, Any, List, Tuple, Optional
import concurrent.futures
import numpy as np
import time
import random
import json

# Task5 Function Definitions for Function Calling
# task5_tools.py - 투자 판단 분석 도구 (수정된 함수 정의)

task5_functions = [
    {
        "name": "get_info_stock",
        "description": "사용자 질문에서 종목명을 기반으로 종목 코드 및 관련 정보를 반환합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "stock_name": {
                    "type": "string",
                    "description": "종목명 (예: 삼성전자, 카카오, Apple)"
                },
                "user_query": {
                    "type": "string",
                    "description": "사용자가 입력한 자연어 질문"
                }
            },
            "required": ["stock_name", "user_query"]
        }
    },
    {
        "name": "get_yfinance_data",
        "description": "Yahoo Finance API를 통해 기업의 시세, 재무제표, 애널리스트 의견, 뉴스 등을 수집합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "ticker": {
                    "type": "string",
                    "description": "Yahoo Finance 티커 코드 (예: '005930.KS', 'AAPL')"
                },
                "data_types": {
                    "type": "array",
                    "items": {
                        "type": "string",
                        "enum": ["price_data", "financial_statements", "analyst_info", "news"]
                    },
                    "default": ["price_data", "financial_statements", "analyst_info", "news"],
                    "description": "수집할 데이터 유형 선택"
                },
                "period": {
                    "type": "string",
                    "enum": ["1mo", "3mo", "6mo", "1y", "2y"],
                    "default": "1y",
                    "description": "주가 데이터 수집 기간"
                }
            },
            "required": ["ticker"]
        }
    },
    {
        "name": "get_dart_disclosure_data",
        "description": "DART 전자공시시스템에서 기업의 재무제표, 감사보고서, 공시 정보 등을 수집합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "corp_name": {
                    "type": "string",  
                    "description": "기업명 (예: 삼성전자, 카카오)"
                },
                "corp_code": {
                    "type": "string",
                    "description": "DART 기업 코드 (8자리)"
                },
                "data_types": {
                    "type": "array",
                    "items": {
                        "type": "string",
                        "enum": ["company_info", "financial_summary", "audit_report", "major_shareholders", "dividends", "employees", "shareholders"]
                    },
                    "default": ["financial_summary"],
                    "description": "수집할 공시 데이터 유형"
                },
                "bsns_year": {
                    "type": "string",
                    "default": "2023",
                    "description": "사업연도 (YYYY 형식)"
                },
                "reprt_code": {
                    "type": "string",
                    "enum": ["11013", "11012", "11014", "11011"],
                    "default": "11011",
                    "description": "보고서 코드: 11011(사업보고서), 11012(반기보고서), 11013(1분기보고서), 11014(3분기보고서)"
                }
            },
            "required": ["corp_name", "corp_code"]
        }
    },
    {
        "name": "calculate_roe_peg",
        "description": "Yahoo Finance 데이터를 기반으로 ROE(자기자본이익률)와 PEG(Price/Earnings to Growth) 비율을 계산합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "fin_data": {
                    "type": "object",
                    "description": "Yahoo Finance에서 수집한 재무 데이터"
                }
            },
            "required": ["fin_data"]
        }
    },
    {
        "name": "format_growth_prompt",
        "description": "성장성 분석에 필요한 주요 데이터를 LLM 프롬프트용 요약 텍스트로 변환합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "fin_data": {
                    "type": "object",
                    "description": "시가총액, PER, PBR, EPS, 배당률 등 주요지표 dict"
                },
                "income_stmt": {
                    "type": "object",
                    "description": "손익계산서 등 재무 데이터 (선택사항)"
                },
                "news_list": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "관련 뉴스 요약문 리스트 (선택사항)"
                },
                "roe": {
                    "type": "number",
                    "description": "자기자본이익률 (선택사항)"
                },
                "peg": {
                    "type": "number", 
                    "description": "PEG 비율 (선택사항)"
                }
            },
            "required": ["fin_data"]
        }
    },
    {
        "name": "format_financial_safety_prompt",
        "description": "DART 수집 데이터에서 재무건전성 판단에 필요한 주요 지표를 추출하고, 체계적 분석을 위한 프롬프트를 생성합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "dart_data": {
                    "type": "object",
                    "description": "DART에서 수집한 공시 데이터 (get_dart_disclosure_data 결과의 data 부분)"
                }
            },
            "required": ["dart_data"]
        }
    },
    {
        "name": "generate_investment_prompt_summary",
        "description": "재무건전성과 성장성 분석 결과를 종합하여 최종 투자 판단 프롬프트를 생성합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "safety_summary": {
                    "type": "string",
                    "description": "format_financial_safety_prompt() 결과 (선택사항)"
                },
                "growth_summary": {
                    "type": "string", 
                    "description": "format_growth_prompt() 결과 (선택사항)"
                },
                "ticker": {
                    "type": "string",
                    "description": "티커 심볼 (선택사항)"
                },
                "stock_name": {
                    "type": "string",
                    "description": "종목명 (선택사항)"
                },
                "base_date": {
                    "type": "string",
                    "description": "기준일 (선택사항)"
                }
            },
            "required": []
        }
    }
]

In [2]:
def get_info_stock(stock_name: str,user_query: str):

    ticker = name_to_ticker(stock_name)
    corp_code = get_corp_code(stock_name)

    return {
        "stock_name": stock_name,
        "ticker": ticker,
        "corp_code": corp_code,
        "original_query": user_query
    }

In [3]:
result = get_info_stock(
    stock_name="카카오",
    user_query="작년 5월에 카카오 산 건 어땠을까?"
)

print(result)

{'stock_name': '카카오', 'ticker': '035720.KS', 'corp_code': '00258801', 'original_query': '작년 5월에 카카오 산 건 어땠을까?'}


## 2. 성장성 데이터 수집 & 처리 모듈

In [36]:
#---------------------------------------------------------------------------
# 2. get_yfinance_data(): Yahoo Finance에서 성장성 지표 수집 (시세, 재무제표, 애널리스트 의견, 뉴스)
#---------------------------------------------------------------------------

def get_yfinance_data(ticker: str, data_types: List[str] = None, period: str = "1y") -> Dict[str, Any]:
    """
    Yahoo Finance에서 종합 데이터 수집
    """
    if data_types is None:
        data_types = ["price_data", "financial_statements", "analyst_info", "news"]
    
    try:
        stock = yf.Ticker(ticker)
        result = {"ticker": ticker, "data": {}}
        
        # 기본 정보
        result["data"]["info"] = stock.info
        
        # 가격 데이터
        if "price_data" in data_types:
            result["data"]["price_data"] = {
                "current_price": stock.info.get("currentPrice"),
                "previous_close": stock.info.get("previousClose"),
                "day_high": stock.info.get("dayHigh"),
                "day_low": stock.info.get("dayLow"),
                "52_week_high": stock.info.get("fiftyTwoWeekHigh"),
                "52_week_low": stock.info.get("fiftyTwoWeekLow"),
                "market_cap": stock.info.get("marketCap"),
                "volume": stock.info.get("volume"),
                "avg_volume": stock.info.get("averageVolume")
            }
            
            # 주가 히스토리
            hist = stock.history(period=period)
            result["data"]["price_history"] = hist.to_dict()
        
        # 재무제표
        if "financial_statements" in data_types:
            result["data"]["financials"] = {
                "income_statement": stock.financials.to_dict() if not stock.financials.empty else {},
                "balance_sheet": stock.balance_sheet.to_dict() if not stock.balance_sheet.empty else {},
                "cash_flow": stock.cashflow.to_dict() if not stock.cashflow.empty else {}
            }
        
        # 애널리스트 정보
        if "analyst_info" in data_types:
            result["data"]["analyst_info"] = {
                "recommendations": stock.recommendations.to_dict() if stock.recommendations is not None else {},
                "target_price": stock.info.get("targetMeanPrice"),
                "recommendation_key": stock.info.get("recommendationKey"),
                "number_of_analysts": stock.info.get("numberOfAnalystOpinions")
            }
        
        # 뉴스
        if "news" in data_types:
            news = stock.news[:5]  # 최근 5개 뉴스만
            result["data"]["news"] = [
                {
                    "title": item.get("title"),
                    "publisher": item.get("publisher"),
                    "published_at": datetime.fromtimestamp(item.get("providerPublishTime", 0)).strftime("%Y-%m-%d %H:%M"),
                    "url": item.get("link")
                } for item in news
            ]
        
        return {"status": "success", "data": result}
        
    except Exception as e:
        return {
            "status": "error", 
            "message": f"Yahoo Finance 데이터 수집 오류: {str(e)}"
        }

In [ ]:
## 함수 test

ticker = "005930.KS"  
data_types = ["price_data", "financial_statements", "analyst_info", "news"]
period = "6mo"

result = get_yfinance_data(ticker, data_types, period)

result

{'status': 'success',
 'data': {'ticker': '005930.KS',
  'data': {'info': {'address1': '129 Samsung-Ro',
    'address2': 'Maetan-3dong Yeongtong-gu',
    'city': 'Suwon',
    'country': 'South Korea',
    'website': 'https://www.samsung.com',
    'industry': 'Consumer Electronics',
    'industryKey': 'consumer-electronics',
    'industryDisp': 'Consumer Electronics',
    'sector': 'Technology',
    'sectorKey': 'technology',
    'sectorDisp': 'Technology',
    'longBusinessSummary': 'Samsung Electronics Co., Ltd. engages in the consumer electronics, information technology and mobile communications, and device solutions businesses worldwide. It operates in four divisions such as DX, DS, SDC and Harman. The company offers smartphones, tablets, audio sounds, watches, switches, and accessories; TVs, and sound devices; appliances, including refrigerators, washing machines and dryers, vacuum cleaners, cooking appliances, dishwashers, air conditioners, and air purifiers; monitors and memory s

In [55]:
def calculate_roe_peg(fin_data: Dict[str, Any]) -> Dict[str, str]:
    """
    ROE 및 PEG 계산 (Yahoo Finance 기반 재무제표 및 애널리스트 데이터 사용)
    """
    summary = {}

    try:
        # 1. ROE 계산: ROE = Net Income / Total Equity * 100
        income_stmt = fin_data.get("financials", {}).get("income_statement", {})
        balance_sheet = fin_data.get("financials", {}).get("balance_sheet", {})
        
        net_income_val = None
        total_equity_val = None

        if income_stmt.get("Net Income"):
            # 가장 최근 날짜 값 사용
            net_income_val = list(income_stmt["Net Income"].values())[0]  # dict[str, float]
        if balance_sheet.get("Total Stockholder Equity"):
            total_equity_val = list(balance_sheet["Total Stockholder Equity"].values())[0]

        if isinstance(net_income_val, (int, float)) and isinstance(total_equity_val, (int, float)) and total_equity_val != 0:
            roe = (net_income_val / total_equity_val) * 100
            summary["ROE"] = f"{roe:.2f}%"
        else:
            summary["ROE"] = "N/A"

        # 2. PEG 계산: PEG = PER / EPS Growth Rate
        per = fin_data.get("info", {}).get("trailingPE")  # 또는 forwardPE
        eps_growth = fin_data.get("info", {}).get("earningsGrowth")  # ex: 0.15 (15%)

        if isinstance(per, (int, float)) and isinstance(eps_growth, (int, float)) and eps_growth != 0:
            peg = per / (eps_growth * 100)  # % 기준으로 나누기
            summary["PEG"] = f"{peg:.2f}"
        else:
            summary["PEG"] = "N/A"

    except Exception as e:
        summary["ROE"] = "N/A"
        summary["PEG"] = "N/A"

    return summary


In [64]:
ticker = "005930.KS"  # 삼성전자
data_types = ["price_data", "financial_statements", "analyst_info", "news"]
period = "6mo"
result = get_yfinance_data(ticker, data_types, period)
calculate_roe_peg(result["data"]["data"])

{'ROE': 'N/A', 'PEG': 'N/A'}

In [ ]:
## LLM전달용 데이터 정리
from typing import Dict, Any, List, Optional

def format_growth_prompt(fin_data: Dict[str, Any], 
                        income_stmt: Optional[Dict[str, Any]]=None, 
                        news_list: Optional[List[str]]=None,roe=None, peg=None) -> str:
    """
    성장성 분석에 필요한 주요 데이터를 LLM 프롬프트용 요약 텍스트로 변환
    - fin_data: 시가총액, PER, PBR, EPS, 배당률 등 주요지표 dict
    - income_stmt: 손익계산서 등 재무 데이터(옵션)
    - news_list: 관련 뉴스 요약문 리스트(옵션)
    """
  

    roe_peg_summary = calculate_roe_peg(fin_data)

    # summary_table에 반영
    roe = roe_peg_summary.get("ROE", "N/A")
    peg = roe_peg_summary.get("PEG", "N/A")


    # 주요 수치 지표 요약
    prompt = []
    prompt.append(f"\n■ 성장성 주요 지표 요약")
    if "현재가" in fin_data: prompt.append(f"- 현재가: {fin_data['현재가']}")
    if "시가총액" in fin_data: prompt.append(f"- 시가총액: {fin_data['시가총액']}")
    if "PER (Trailing)" in fin_data: prompt.append(f"- PER: {fin_data['PER (Trailing)']}")
    if "PBR" in fin_data: prompt.append(f"- PBR: {fin_data['PBR']}")
    if "EPS" in fin_data: prompt.append(f"- EPS: {fin_data.get('EPS', '정보 없음')}")
    if "배당률" in fin_data: prompt.append(f"- 배당률: {fin_data['배당률']}")
    if "추천등급" in fin_data: prompt.append(f"- 애널리스트 평균 등급: {fin_data['추천등급']}")
    if "목표가 평균" in fin_data: prompt.append(f"- 애널리스트 목표가: {fin_data['목표가 평균']}")
    
    # 손익계산서 등 성장 관련 실적
    if income_stmt:
        rev = income_stmt.get("매출액") or income_stmt.get("Revenue")
        op_profit = income_stmt.get("영업이익") or income_stmt.get("Operating Income")
        net_profit = income_stmt.get("당기순이익") or income_stmt.get("Net Income")
        year = income_stmt.get("연도", "")
        if rev or op_profit or net_profit:
            prompt.append(f"\n■ 최근 재무 성과")
            if year: prompt.append(f"- 연도: {year}")
            if rev: prompt.append(f"- 매출액: {rev}")
            if op_profit: prompt.append(f"- 영업이익: {op_profit}")
            if net_profit: prompt.append(f"- 순이익: {net_profit}")
    
    # 재무제표 요약
    print("\n■ 재무제표 요약")
    fs = data.get("financials", {})
    for sheet_name in ["income_statement", "balance_sheet", "cash_flow"]:
        print(f"\n{sheet_name.replace('_', ' ').title()}")
        sheet = fs.get(sheet_name, {})
        if sheet:
            first_item = list(sheet.items())[0] if sheet else ("N/A", {})
            metric, values = first_item
            for date, value in list(values.items())[:3]:
                if isinstance(value, (int, float)):
                    print(f"- {metric} | {date[:10]}: {value/1e9:.2f}B USD")
        else:
            print("- 데이터 없음")
        
    # 관련 뉴스 헤드라인 요약 추가
    if news_list:
        prompt.append(f"\n■ 최근 시장 뉴스 요약")
        for news in news_list[:3]:  # 최대 3개만 예시
            prompt.append(f"- {news}")

    if roe:
        prompt.append(f"- ROE(자기자본이익률): {roe}%")
    if peg is not None:
        prompt.append(f"- PEG(성장반영 PER): {peg}")

    # 이론 및 평가 기준 요약 포함
    prompt.append("\n■ 투자매력 평가기준 안내")
    prompt.append("- 매출, 이익이 최근 몇 년간 꾸준히 성장했는가?")
    prompt.append("- ROE 등 수익성 지표가 업계 평균 대비 우수한가?")
    prompt.append("- PER, PBR 등 밸류에이션이 저평가/고평가 구간인지?")
    prompt.append("- 애널리스트 목표가·뉴스 등 시장의 기대/우려 요인은?")
    prompt.append("\n상기 계량적 지표와 기준을 참고해 기업의 성장성과 투자매력도를 논리적으로 종합 판단해줘.")
    return "\n".join(prompt)


In [62]:
ticker = "005930.KS"  # 삼성전자
data_types = ["price_data", "financial_statements", "analyst_info", "news"]
period = "6mo"
result = get_yfinance_data(ticker, data_types, period)


growth_prompt = format_growth_prompt(
            fin_data=summary_table,
            # news_list=news_titles,
            # roe=12.5,
            # peg=1.2
        )
        
print(growth_prompt)


■ 재무제표 요약

Income Statement
- 2024-12-31 00:00:00 | Tax Effect: -7.64B USD
- 2024-12-31 00:00:00 | Tax Rate F: 0.00B USD
- 2024-12-31 00:00:00 | Normalized: 81157.66B USD

Balance Sheet
- 2024-12-31 00:00:00 | Treasury S: nanB USD
- 2024-12-31 00:00:00 | Preferred : 0.82B USD
- 2024-12-31 00:00:00 | Ordinary S: 6.79B USD

Cash Flow
- 2024-12-31 00:00:00 | Free Cash : 19240.98B USD
- 2024-12-31 00:00:00 | Repurchase: -1811.78B USD
- 2024-12-31 00:00:00 | Repayment : -1364.51B USD

■ 성장성 주요 지표 요약
- 현재가: 71400.0 USD
- 시가총액: 469.77조 USD
- PER: N/A
- PBR: N/A
- 배당률: 200.00%
- 애널리스트 평균 등급: buy
- 애널리스트 목표가: 76295.15 USD
- ROE(자기자본이익률): N/A%
- PEG(성장반영 PER): N/A

■ 투자매력 평가기준 안내
- 매출, 이익이 최근 몇 년간 꾸준히 성장했는가?
- ROE 등 수익성 지표가 업계 평균 대비 우수한가?
- PER, PBR 등 밸류에이션이 저평가/고평가 구간인지?
- 애널리스트 목표가·뉴스 등 시장의 기대/우려 요인은?

상기 계량적 지표와 기준을 참고해 기업의 성장성과 투자매력도를 논리적으로 종합 판단해 한글로 요약해줘.


## 3. 재무 건정성 평가 (DART API) 데이터 수집& 처리 모듈

In [ ]:
from typing import List, Dict, Any
import pandas as pd
import dart_fss

def get_dart_disclosure_data(corp_name: str, corp_code: str,
                              data_types: List[str] = None,
                              bsns_year: str = "2023",
                              reprt_code: str = "11011") -> Dict[str, Any]:
    """
    DART 전자공시 Open API를 활용해 기업 정보, 재무제표, 감사보고서 등 수집
    """
    if data_types is None:
        data_types = ["company_info", "financial_summary", "audit_report"]

    try:
        # API 키 설정
        api_key = '0778fdc977a04c6bc0516326fe22cea0224b233b'
        dart_fss.set_api_key(api_key)
        
        result = {"corp_name": corp_name, "corp_code": corp_code, "data": {}}

        # 1. 기업 개요
        if "company_info" in data_types:
            info = dart_fss.api.filings.get_corp_info(corp_code)
            result["data"]["company_info"] = info

        # 2. 재무제표 요약
        if "financial_summary" in data_types:
            fin = dart_fss.api.finance.fnltt_singl_acnt(corp_code, bsns_year, reprt_code)
            result["data"]["financial_summary"] = pd.DataFrame(fin['list']) if 'list' in fin else {}

        # 3. 감사보고서
        if "audit_report" in data_types:
            try:
                report = dart_fss.api.finance.audit_report(corp_code, bsns_year, reprt_code)
                result["data"]["audit_report"] = pd.DataFrame(report['list']) if 'list' in report else {}
            except:
                result["data"]["audit_report"] = {}

        # 4. 최대주주 현황
        if "major_shareholders" in data_types:
            holders = dart_fss.api.info.hyslr_sttus(corp_code, bsns_year, reprt_code)
            result["data"]["major_shareholders"] = pd.DataFrame(holders['list']) if 'list' in holders else {}

        # 5. 배당 현황
        if "dividends" in data_types:
            dividends = dart_fss.api.info.alot_matter(corp_code, bsns_year, reprt_code)
            result["data"]["dividends"] = pd.DataFrame(dividends['list']) if 'list' in dividends else {}

        # 6. 직원 현황
        if "employees" in data_types:
            emp = dart_fss.api.info.emp_sttus(corp_code, bsns_year, reprt_code)
            result["data"]["employees"] = pd.DataFrame(emp['list']) if 'list' in emp else {}

        # 7. 주주 현황
        if "shareholders" in data_types:
            shareholders = dart_fss.api.shareholder.elestock(corp_code)
            result["data"]["shareholders"] = pd.DataFrame(shareholders['list']) if 'list' in shareholders else {}

        return {"status": "success", "data": result}

    except Exception as e:
        return {
            "status": "error",
            "message": f"DART 데이터 수집 오류: {str(e)}"
        }


In [26]:
result = get_dart_disclosure_data(corp_name="삼성전자", corp_code='00126380')

if result["status"] == "success":
    from pprint import pprint
    pprint(result["data"].keys())  # 수집된 파트 확인
    print(result)
else:
    print("❌", result["message"])

dict_keys(['corp_name', 'corp_code', 'data'])
{'status': 'success', 'data': {'corp_name': '삼성전자', 'corp_code': '00126380', 'data': {'financial_summary':           rcept_no reprt_code bsns_year corp_code stock_code fs_div   fs_nm  \
0   20240312000736      11011      2023  00126380     005930    CFS  연결재무제표   
1   20240312000736      11011      2023  00126380     005930    CFS  연결재무제표   
2   20240312000736      11011      2023  00126380     005930    CFS  연결재무제표   
3   20240312000736      11011      2023  00126380     005930    CFS  연결재무제표   
4   20240312000736      11011      2023  00126380     005930    CFS  연결재무제표   
5   20240312000736      11011      2023  00126380     005930    CFS  연결재무제표   
6   20240312000736      11011      2023  00126380     005930    CFS  연결재무제표   
7   20240312000736      11011      2023  00126380     005930    CFS  연결재무제표   
8   20240312000736      11011      2023  00126380     005930    CFS  연결재무제표   
9   20240312000736      11011      2023  00126380     005

In [20]:
# 1. 성공 여부 확인
if result["status"] == "success":
    # 2. 재무제표 데이터 꺼내기
    fin_df = result["data"]["data"].get("financial_summary", pd.DataFrame())

    # 3. 컬럼 출력
    if isinstance(fin_df, pd.DataFrame):
        print(fin_df.columns.tolist())
    else:
        print("📭 재무제표 데이터 없음")
else:
    print(result["message"])

['rcept_no', 'reprt_code', 'bsns_year', 'corp_code', 'stock_code', 'fs_div', 'fs_nm', 'sj_div', 'sj_nm', 'account_nm', 'thstrm_nm', 'thstrm_dt', 'thstrm_amount', 'frmtrm_nm', 'frmtrm_dt', 'frmtrm_amount', 'bfefrmtrm_nm', 'bfefrmtrm_dt', 'bfefrmtrm_amount', 'ord', 'currency']


In [66]:
from typing import Dict, Any
import pandas as pd

def format_financial_safety_prompt(dart_data: Dict[str, Any]) -> str:
    """
    DART 수집 데이터에서 재무건전성 판단에 필요한 주요 지표를 추출하고,
    프롬프트용 요약 텍스트로 변환

    dart_data: get_dart_disclosure_data()의 result["data"]["data"] 값
    """
    prompt = ["\n■ 재무 건전성 분석 요약"]

    try:
        fin_df = dart_data.get("financial_summary", pd.DataFrame())

        if not isinstance(fin_df, pd.DataFrame) or fin_df.empty:
            prompt.append("- 재무제표 요약 데이터가 없습니다.")
            return "\n".join(prompt)

        # 유의미한 계정 항목 필터링
        targets = [
            "유동비율", "부채비율", "차입금의존도", "유보율", "이자보상배율",
            "당기순이익", "영업이익", "이자비용", "자산총계", "부채총계", "자본총계"
        ]
        fin_df = fin_df[fin_df["account_nm"].isin(targets)]

        # 주요 지표 출력
        grouped = fin_df.groupby("account_nm").first().reset_index()
        for _, row in grouped.iterrows():
            acc = row["account_nm"]
            val = row["thstrm_amount"]
            try:
                val = f"{int(val):,} 원" if val and str(val).isdigit() else val
            except:
                pass
            prompt.append(f"- {acc}: {val}")

        # 평가 기준 안내
        prompt.append("\n■ 재무건전성 평가기준 안내")
        prompt.append("- 유동비율 100% 이상: 단기지급능력 양호")
        prompt.append("- 부채비율 200% 이하: 안정성 확보")
        prompt.append("- 차입금의존도 30% 이하: 외부자본 의존도 낮음")
        prompt.append("- 이자보상배율 1 이상: 이자 지급능력 확보")
        prompt.append("- 당기순손실 지속 시 유의")

        prompt.append("\n상기 지표와 기준을 종합해 기업의 재무 건전성을 평가해줘.")
        return "\n".join(prompt)

    except Exception as e:
        return f"❌ 재무건전성 분석 중 오류: {str(e)}"

In [82]:
result = get_dart_disclosure_data(corp_name="카카오",corp_code='00258801' ,data_types=["financial_summary"])
if result["status"] == "success":
    dart_data = result["data"]["data"]
    prompt_text1 = format_financial_safety_prompt(dart_data)
    print(prompt_text1)
else:
    print(result["message"])


■ 재무건전성 종합 분석 요약

■ 1. 유동성 분석 (Liquidity Analysis)
- 유동비율: 0.0% (유동자산/유동부채)
- 당좌비율: 0.0% ((유동자산-재고자산)/유동부채)
- 현금비율: 0.0% (현금성자산/유동부채)
- 순운전자본: 0원 (유동자산-유동부채)

■ 2. 활동성 분석 (Activity Analysis)
- 총자산회전율: 0.00회 (매출액/총자산)

■ 3. 레버리지 분석 (Leverage Analysis)
- 부채비율: 0.0% (총부채/총자본)
- 부채대자산비율: 0.0% (총부채/총자산)
- 이자보상배율: 무한대 (이자비용 없음)
- 차입금의존도: 0.0% (차입금/총자산)
- 자기자본비율: 0.0% (총자본/총자산)

■ 4. 수익성 분석 (Profitability Analysis)
- 영업이익률: 0.0% (영업이익/매출액)
- 순이익률: 0.0% (당기순이익/매출액)
- ROA: 0.0% (당기순이익/총자산)
- ROE: 0.0% (당기순이익/총자본)

■ 5. 성장성 분석 (Growth Analysis)
- 성장성 분석을 위해서는 전년도 데이터와의 비교가 필요합니다.
- 매출액 성장률, 영업이익 성장률, 자산 성장률 등을 평가하세요.

■ 7. 신용위험 평가 (Credit Risk Assessment)
- Altman Z-Score: 0.00
  → 위험 구간 (부도위험 높음)

■ 8. 재무건전성 평가 기준

【유동성 기준】
- 매우우수: 유동비율 200%↑, 당좌비율 100%↑
- 우수: 유동비율 150-200%, 당좌비율 80-100%
- 보통: 유동비율 100-150%, 당좌비율 50-80%
- 주의: 유동비율 80-100%, 당좌비율 30-50%
- 위험: 유동비율 80%↓, 당좌비율 30%↓

【레버리지 기준】
- 매우우수: 부채비율 100%↓, 이자보상배율 10배↑
- 우수: 부채비율 100-150%, 이자보상배율 5-10배
- 보통: 부채비율 150-200%, 이자보상배율 2-5배
- 주의: 부

In [76]:
from typing import Dict, Any
import pandas as pd
import numpy as np


def format_financial_safety_prompt(dart_data: Dict[str, Any]) -> str:
    """
    DART 수집 데이터에서 재무건전성 판단에 필요한 주요 지표를 추출하고,
    금융 전문가 수준의 체계적 분석을 위한 프롬프트 생성
    
    Args:
        dart_data: get_dart_disclosure_data()의 result["data"] 값
    
    Returns:
        str: 재무건전성 분석용 프롬프트 텍스트
    """
    prompt = ["\n■ 재무건전성 종합 분석 요약"]
    
    try:
        fin_df = dart_data.get("financial_summary", pd.DataFrame())
        shareholders_df = dart_data.get("shareholders", pd.DataFrame())
        
        if not isinstance(fin_df, pd.DataFrame) or fin_df.empty:
            prompt.append("- 재무제표 요약 데이터가 없습니다.")
            return "\n".join(prompt)
        
        # 재무제표 데이터 전처리
        fin_dict = _process_financial_data(fin_df)
        
        # 1. 유동성 분석 (Liquidity Analysis)
        prompt.extend(_analyze_liquidity(fin_dict))
        
        # 2. 활동성 분석 (Activity Analysis)  
        prompt.extend(_analyze_activity(fin_dict))
        
        # 3. 레버리지 분석 (Leverage Analysis)
        prompt.extend(_analyze_leverage(fin_dict))
        
        # 4. 수익성 분석 (Profitability Analysis)
        prompt.extend(_analyze_profitability(fin_dict))
        
        # 5. 성장성 분석 (Growth Analysis)
        prompt.extend(_analyze_growth(fin_dict))
        
        # 6. 지배구조 분석 (Corporate Governance)
        if not shareholders_df.empty:
            prompt.extend(_analyze_governance(shareholders_df))
        
        # 7. 신용위험 평가 (Credit Risk Assessment)
        prompt.extend(_assess_credit_risk(fin_dict))
        
        # 8. 종합 평가 기준
        prompt.extend(_comprehensive_evaluation_criteria())
        
        prompt.append("\n상기 재무지표와 평가기준을 바탕으로 기업의 재무건전성을 5단계(매우우수/우수/보통/주의/위험)로 평가하고, 각 영역별 강점과 개선점을 제시해주세요.")
        
        return "\n".join(prompt)
        
    except Exception as e:
        return f"❌ 재무건전성 분석 중 오류: {str(e)}"


def _process_financial_data(fin_df: pd.DataFrame) -> Dict[str, float]:
    """재무제표 데이터를 딕셔너리로 변환"""
    fin_dict = {}
    
    # 주요 계정과목 매핑
    account_mapping = {
        # 자산
        "유동자산": ["유동자산"],
        "비유동자산": ["비유동자산"],
        "자산총계": ["자산총계"],
        "현금및현금성자산": ["현금및현금성자산", "현금및단기금융상품"],
        "매출채권": ["매출채권", "받을어음및매출채권"],
        "재고자산": ["재고자산"],
        
        # 부채
        "유동부채": ["유동부채"],
        "비유동부채": ["비유동부채"],
        "부채총계": ["부채총계"],
        "차입금": ["단기차입금", "장기차입금", "사채"],
        "이자비용": ["이자비용"],
        
        # 자본
        "자본총계": ["자본총계"],
        "자본금": ["자본금"],
        "이익잉여금": ["이익잉여금"],
        
        # 손익
        "매출액": ["매출액", "수익(매출액)"],
        "영업이익": ["영업이익"],
        "당기순이익": ["당기순이익"],
        "법인세비용": ["법인세비용"],
        "EBITDA": ["EBITDA"],
        "감가상각비": ["감가상각비"]
    }
    
    for key, account_names in account_mapping.items():
        for account_name in account_names:
            matching_rows = fin_df[fin_df["account_nm"] == account_name]
            if not matching_rows.empty:
                value = matching_rows.iloc[0]["thstrm_amount"]
                try:
                    fin_dict[key] = float(value) if value and str(value).replace('-','').replace('.','').isdigit() else 0
                    break
                except:
                    fin_dict[key] = 0
        
        if key not in fin_dict:
            fin_dict[key] = 0
    
    return fin_dict


def _analyze_liquidity(fin_dict: Dict[str, float]) -> list:
    """유동성 분석"""
    analysis = ["\n■ 1. 유동성 분석 (Liquidity Analysis)"]
    
    # 유동비율 (Current Ratio)
    current_ratio = (fin_dict["유동자산"] / fin_dict["유동부채"] * 100) if fin_dict["유동부채"] != 0 else 0
    analysis.append(f"- 유동비율: {current_ratio:.1f}% (유동자산/유동부채)")
    
    # 당좌비율 (Quick Ratio)
    quick_assets = fin_dict["유동자산"] - fin_dict["재고자산"]
    quick_ratio = (quick_assets / fin_dict["유동부채"] * 100) if fin_dict["유동부채"] != 0 else 0
    analysis.append(f"- 당좌비율: {quick_ratio:.1f}% ((유동자산-재고자산)/유동부채)")
    
    # 현금비율 (Cash Ratio)
    cash_ratio = (fin_dict["현금및현금성자산"] / fin_dict["유동부채"] * 100) if fin_dict["유동부채"] != 0 else 0
    analysis.append(f"- 현금비율: {cash_ratio:.1f}% (현금성자산/유동부채)")
    
    # 순운전자본 (Net Working Capital)
    nwc = fin_dict["유동자산"] - fin_dict["유동부채"]
    analysis.append(f"- 순운전자본: {nwc:,.0f}원 (유동자산-유동부채)")
    
    return analysis


def _analyze_activity(fin_dict: Dict[str, float]) -> list:
    """활동성 분석"""
    analysis = ["\n■ 2. 활동성 분석 (Activity Analysis)"]
    
    # 총자산회전율 (Total Asset Turnover)
    asset_turnover = fin_dict["매출액"] / fin_dict["자산총계"] if fin_dict["자산총계"] != 0 else 0
    analysis.append(f"- 총자산회전율: {asset_turnover:.2f}회 (매출액/총자산)")
    
    # 매출채권회전율 (Receivables Turnover)
    if fin_dict["매출채권"] != 0:
        receivable_turnover = fin_dict["매출액"] / fin_dict["매출채권"]
        receivable_days = 365 / receivable_turnover if receivable_turnover != 0 else 0
        analysis.append(f"- 매출채권회전율: {receivable_turnover:.1f}회, 회수기간: {receivable_days:.0f}일")
    
    # 재고자산회전율 (Inventory Turnover)
    if fin_dict["재고자산"] != 0:
        inventory_turnover = fin_dict["매출액"] / fin_dict["재고자산"]
        inventory_days = 365 / inventory_turnover if inventory_turnover != 0 else 0
        analysis.append(f"- 재고자산회전율: {inventory_turnover:.1f}회, 보유기간: {inventory_days:.0f}일")
    
    return analysis


def _analyze_leverage(fin_dict: Dict[str, float]) -> list:
    """레버리지 분석"""
    analysis = ["\n■ 3. 레버리지 분석 (Leverage Analysis)"]
    
    # 부채비율 (Debt Ratio)
    debt_ratio = (fin_dict["부채총계"] / fin_dict["자본총계"] * 100) if fin_dict["자본총계"] != 0 else 0
    analysis.append(f"- 부채비율: {debt_ratio:.1f}% (총부채/총자본)")
    
    # 부채대자산비율 (Debt to Asset Ratio)
    debt_to_asset = (fin_dict["부채총계"] / fin_dict["자산총계"] * 100) if fin_dict["자산총계"] != 0 else 0
    analysis.append(f"- 부채대자산비율: {debt_to_asset:.1f}% (총부채/총자산)")
    
    # 이자보상배율 (Interest Coverage Ratio)
    interest_coverage = fin_dict["영업이익"] / fin_dict["이자비용"] if fin_dict["이자비용"] != 0 else float('inf')
    if interest_coverage == float('inf'):
        analysis.append("- 이자보상배율: 무한대 (이자비용 없음)")
    else:
        analysis.append(f"- 이자보상배율: {interest_coverage:.1f}배 (영업이익/이자비용)")
    
    # 차입금의존도 (Borrowings Dependency)
    borrowing_dependency = (fin_dict["차입금"] / fin_dict["자산총계"] * 100) if fin_dict["자산총계"] != 0 else 0
    analysis.append(f"- 차입금의존도: {borrowing_dependency:.1f}% (차입금/총자산)")
    
    # 자기자본비율 (Equity Ratio)
    equity_ratio = (fin_dict["자본총계"] / fin_dict["자산총계"] * 100) if fin_dict["자산총계"] != 0 else 0
    analysis.append(f"- 자기자본비율: {equity_ratio:.1f}% (총자본/총자산)")
    
    return analysis


def _analyze_profitability(fin_dict: Dict[str, float]) -> list:
    """수익성 분석"""
    analysis = ["\n■ 4. 수익성 분석 (Profitability Analysis)"]
    
    # 매출액영업이익률 (Operating Margin)
    operating_margin = (fin_dict["영업이익"] / fin_dict["매출액"] * 100) if fin_dict["매출액"] != 0 else 0
    analysis.append(f"- 영업이익률: {operating_margin:.1f}% (영업이익/매출액)")
    
    # 매출액순이익률 (Net Margin)
    net_margin = (fin_dict["당기순이익"] / fin_dict["매출액"] * 100) if fin_dict["매출액"] != 0 else 0
    analysis.append(f"- 순이익률: {net_margin:.1f}% (당기순이익/매출액)")
    
    # 총자산이익률 (ROA)
    roa = (fin_dict["당기순이익"] / fin_dict["자산총계"] * 100) if fin_dict["자산총계"] != 0 else 0
    analysis.append(f"- ROA: {roa:.1f}% (당기순이익/총자산)")
    
    # 자기자본이익률 (ROE)
    roe = (fin_dict["당기순이익"] / fin_dict["자본총계"] * 100) if fin_dict["자본총계"] != 0 else 0
    analysis.append(f"- ROE: {roe:.1f}% (당기순이익/총자본)")
    
    # EBITDA 마진
    if fin_dict["EBITDA"] != 0:
        ebitda_margin = (fin_dict["EBITDA"] / fin_dict["매출액"] * 100) if fin_dict["매출액"] != 0 else 0
        analysis.append(f"- EBITDA 마진: {ebitda_margin:.1f}% (EBITDA/매출액)")
    
    return analysis


def _analyze_growth(fin_dict: Dict[str, float]) -> list:
    """성장성 분석 (전년대비 데이터가 있을 경우)"""
    analysis = ["\n■ 5. 성장성 분석 (Growth Analysis)"]
    analysis.append("- 성장성 분석을 위해서는 전년도 데이터와의 비교가 필요합니다.")
    analysis.append("- 매출액 성장률, 영업이익 성장률, 자산 성장률 등을 평가하세요.")
    
    return analysis


def _analyze_governance(shareholders_df: pd.DataFrame) -> list:
    """지배구조 분석"""
    analysis = ["\n■ 6. 지배구조 분석 (Corporate Governance)"]
    
    try:
        # 최대주주 지분율
        if not shareholders_df.empty and 'stkrt' in shareholders_df.columns:
            max_ownership = shareholders_df['stkrt'].max()
            analysis.append(f"- 최대주주 지분율: {max_ownership}%")
            
            # 소유집중도 평가
            if max_ownership > 50:
                analysis.append("- 소유구조: 지배주주 존재 (지분율 50% 초과)")
            elif max_ownership > 30:
                analysis.append("- 소유구조: 상당한 영향력 보유 (지분율 30-50%)")
            else:
                analysis.append("- 소유구조: 분산된 소유구조 (지분율 30% 이하)")
        else:
            analysis.append("- 주주구조 데이터가 불충분합니다.")
            
    except Exception as e:
        analysis.append(f"- 지배구조 분석 오류: {str(e)}")
    
    return analysis


def _assess_credit_risk(fin_dict: Dict[str, float]) -> list:
    """신용위험 평가"""
    analysis = ["\n■ 7. 신용위험 평가 (Credit Risk Assessment)"]
    
    # Altman Z-Score 계산 (상장기업용)
    try:
        # Z = 1.2*X1 + 1.4*X2 + 3.3*X3 + 0.6*X4 + 1.0*X5
        # X1 = 순운전자본/총자산
        # X2 = 이익잉여금/총자산  
        # X3 = 영업이익/총자산
        # X4 = 시가총액/총부채 (여기서는 자기자본/총부채로 대체)
        # X5 = 매출액/총자산
        
        x1 = (fin_dict["유동자산"] - fin_dict["유동부채"]) / fin_dict["자산총계"] if fin_dict["자산총계"] != 0 else 0
        x2 = fin_dict["이익잉여금"] / fin_dict["자산총계"] if fin_dict["자산총계"] != 0 else 0
        x3 = fin_dict["영업이익"] / fin_dict["자산총계"] if fin_dict["자산총계"] != 0 else 0
        x4 = fin_dict["자본총계"] / fin_dict["부채총계"] if fin_dict["부채총계"] != 0 else 0
        x5 = fin_dict["매출액"] / fin_dict["자산총계"] if fin_dict["자산총계"] != 0 else 0
        
        z_score = 1.2*x1 + 1.4*x2 + 3.3*x3 + 0.6*x4 + 1.0*x5
        
        analysis.append(f"- Altman Z-Score: {z_score:.2f}")
        
        if z_score > 2.99:
            analysis.append("  → 안전 구간 (부도위험 낮음)")
        elif z_score > 1.81:
            analysis.append("  → 회색 구간 (주의 필요)")
        else:
            analysis.append("  → 위험 구간 (부도위험 높음)")
            
    except Exception as e:
        analysis.append(f"- Z-Score 계산 오류: {str(e)}")
    
    # 현금흐름 안정성 (데이터가 있는 경우)
    if fin_dict["현금및현금성자산"] > 0:
        cash_to_debt = (fin_dict["현금및현금성자산"] / fin_dict["부채총계"] * 100) if fin_dict["부채총계"] != 0 else 0
        analysis.append(f"- 현금 대비 부채비율: {cash_to_debt:.1f}%")
    
    return analysis


def _comprehensive_evaluation_criteria() -> list:
    """종합 평가 기준"""
    criteria = ["\n■ 8. 재무건전성 평가 기준"]
    
    criteria.extend([
        "\n【유동성 기준】",
        "- 매우우수: 유동비율 200%↑, 당좌비율 100%↑",
        "- 우수: 유동비율 150-200%, 당좌비율 80-100%", 
        "- 보통: 유동비율 100-150%, 당좌비율 50-80%",
        "- 주의: 유동비율 80-100%, 당좌비율 30-50%",
        "- 위험: 유동비율 80%↓, 당좌비율 30%↓",
        
        "\n【레버리지 기준】",
        "- 매우우수: 부채비율 100%↓, 이자보상배율 10배↑",
        "- 우수: 부채비율 100-150%, 이자보상배율 5-10배",
        "- 보통: 부채비율 150-200%, 이자보상배율 2-5배", 
        "- 주의: 부채비율 200-300%, 이자보상배율 1-2배",
        "- 위험: 부채비율 300%↑, 이자보상배율 1배↓",
        
        "\n【수익성 기준】",
        "- 매우우수: ROE 15%↑, ROA 10%↑, 영업이익률 15%↑",
        "- 우수: ROE 10-15%, ROA 5-10%, 영업이익률 10-15%",
        "- 보통: ROE 5-10%, ROA 3-5%, 영업이익률 5-10%",
        "- 주의: ROE 0-5%, ROA 0-3%, 영업이익률 0-5%", 
        "- 위험: ROE 0%↓, ROA 0%↓, 영업손실",
        
        "\n【신용위험 기준】",
        "- Altman Z-Score > 2.99: 안전",
        "- Altman Z-Score 1.81-2.99: 회색지대",
        "- Altman Z-Score < 1.81: 위험"
    ])
    
    return criteria

In [78]:
result = get_dart_disclosure_data(corp_name="카카오",corp_code='00258801')
if result["status"] == "success":
    dart_data = result["data"]["data"]
    prompt_text = format_financial_safety_prompt(dart_data)
    print(prompt_text)
else:
    print(result["message"])


■ 재무건전성 종합 분석 요약

■ 1. 유동성 분석 (Liquidity Analysis)
- 유동비율: 0.0% (유동자산/유동부채)
- 당좌비율: 0.0% ((유동자산-재고자산)/유동부채)
- 현금비율: 0.0% (현금성자산/유동부채)
- 순운전자본: 0원 (유동자산-유동부채)

■ 2. 활동성 분석 (Activity Analysis)
- 총자산회전율: 0.00회 (매출액/총자산)

■ 3. 레버리지 분석 (Leverage Analysis)
- 부채비율: 0.0% (총부채/총자본)
- 부채대자산비율: 0.0% (총부채/총자산)
- 이자보상배율: 무한대 (이자비용 없음)
- 차입금의존도: 0.0% (차입금/총자산)
- 자기자본비율: 0.0% (총자본/총자산)

■ 4. 수익성 분석 (Profitability Analysis)
- 영업이익률: 0.0% (영업이익/매출액)
- 순이익률: 0.0% (당기순이익/매출액)
- ROA: 0.0% (당기순이익/총자산)
- ROE: 0.0% (당기순이익/총자본)

■ 5. 성장성 분석 (Growth Analysis)
- 성장성 분석을 위해서는 전년도 데이터와의 비교가 필요합니다.
- 매출액 성장률, 영업이익 성장률, 자산 성장률 등을 평가하세요.

■ 7. 신용위험 평가 (Credit Risk Assessment)
- Altman Z-Score: 0.00
  → 위험 구간 (부도위험 높음)

■ 8. 재무건전성 평가 기준

【유동성 기준】
- 매우우수: 유동비율 200%↑, 당좌비율 100%↑
- 우수: 유동비율 150-200%, 당좌비율 80-100%
- 보통: 유동비율 100-150%, 당좌비율 50-80%
- 주의: 유동비율 80-100%, 당좌비율 30-50%
- 위험: 유동비율 80%↓, 당좌비율 30%↓

【레버리지 기준】
- 매우우수: 부채비율 100%↓, 이자보상배율 10배↑
- 우수: 부채비율 100-150%, 이자보상배율 5-10배
- 보통: 부채비율 150-200%, 이자보상배율 2-5배
- 주의: 부

## 4. 평가 함수

In [80]:
from typing import Optional

def generate_investment_prompt_summary(
    safety_summary: Optional[str] = None,
    growth_summary: Optional[str] = None,
    ticker: Optional[str] = None,
    stock_name: Optional[str] = None,
    base_date: Optional[str] = None
) -> str:
    """
    최종 투자 판단 프롬프트 요약 생성 (재무건전성, 성장성 요약 결합)
    - safety_summary: format_financial_safety_prompt() 결과
    - growth_summary: format_growth_prompt() 결과
    - ticker / stock_name / base_date: 리포트 메타 정보
    """
    
    header = f"""[Fin문철 종합 리포트] 📈 종목: {stock_name} ({ticker}) 📅 기준일: {base_date or 'N/A'}\n"""
    
    if safety_summary and growth_summary:
        return header + f"""
🔍 <1> 재무 건전성 진단 요약:
{safety_summary}

🚀 <2> 시장 지표 기반 성장성 분석:
{growth_summary}

📌 <AI 종합 판단 요약>
- 해당 종목은 현재 **재무구조의 안정성**과 **시장지표 상 저평가 여부**를 종합적으로 고려할 때, {"상대적으로 안정된 투자처로 평가됨" if "양호" in safety_summary and "저평가" in growth_summary else "투자 시 주의가 요구됨"}.
- 단기 수익보다는 중장기적 관점에서 접근하는 것이 바람직함.
"""
    
    elif safety_summary:
        return header + f"""
🔍 <1> 재무 건전성 진단 요약:
{safety_summary}

📌 <AI 종합 판단 요약>
- 공시 기반 재무구조 평가 결과, 해당 기업은 {"재무적으로 안정된 편" if "양호" in safety_summary or "안정" in safety_summary else "재무 리스크가 존재"}.
- {"재무구조가 튼튼하여 안정적인 투자 기반이 될 수 있음" if "양호" in safety_summary else "현 시점 투자에는 보수적 접근이 필요함"}.
"""
    
    elif growth_summary:
        return header + f"""
🚀 <1> 시장 지표 기반 성장성 분석:
{growth_summary}

📌 <AI 종합 판단 요약>
- 시장 데이터에 따르면 해당 종목은 현재 {"PER/PBR 기준 저평가 상태로 보이며" if "저평가" in growth_summary else "고평가 상태로 보이며"}, {"기대 수익 가능성이 존재" if "저평가" in growth_summary else "신중한 접근이 필요"}.
- {"단기 모멘텀이 강하므로 매수 타이밍 포착이 중요" if "강세" in growth_summary else "시장 흐름 변화에 유의 필요"}.
"""

    else:
        return header + """
⚠️ 데이터 부족
- 재무건전성 및 성장성에 대한 분석 데이터가 부족하여 투자 판단을 내리기 어려움.
- 공시 또는 시장 데이터를 확인 후 다시 분석을 요청하세요.
"""


In [84]:
# 사례: 두 평가 모두 존재
prompt = generate_investment_prompt_summary(
    ticker="005930.KS",
    stock_name="삼성전자",
    base_date="2025-07-31",
    safety_summary=prompt_text1,
    growth_summary=prompt_text,
)
print(prompt)


[Fin문철 종합 리포트] 📈 종목: 삼성전자 (005930.KS) 📅 기준일: 2025-07-31

🔍 <1> 재무 건전성 진단 요약:

■ 재무건전성 종합 분석 요약

■ 1. 유동성 분석 (Liquidity Analysis)
- 유동비율: 0.0% (유동자산/유동부채)
- 당좌비율: 0.0% ((유동자산-재고자산)/유동부채)
- 현금비율: 0.0% (현금성자산/유동부채)
- 순운전자본: 0원 (유동자산-유동부채)

■ 2. 활동성 분석 (Activity Analysis)
- 총자산회전율: 0.00회 (매출액/총자산)

■ 3. 레버리지 분석 (Leverage Analysis)
- 부채비율: 0.0% (총부채/총자본)
- 부채대자산비율: 0.0% (총부채/총자산)
- 이자보상배율: 무한대 (이자비용 없음)
- 차입금의존도: 0.0% (차입금/총자산)
- 자기자본비율: 0.0% (총자본/총자산)

■ 4. 수익성 분석 (Profitability Analysis)
- 영업이익률: 0.0% (영업이익/매출액)
- 순이익률: 0.0% (당기순이익/매출액)
- ROA: 0.0% (당기순이익/총자산)
- ROE: 0.0% (당기순이익/총자본)

■ 5. 성장성 분석 (Growth Analysis)
- 성장성 분석을 위해서는 전년도 데이터와의 비교가 필요합니다.
- 매출액 성장률, 영업이익 성장률, 자산 성장률 등을 평가하세요.

■ 7. 신용위험 평가 (Credit Risk Assessment)
- Altman Z-Score: 0.00
  → 위험 구간 (부도위험 높음)

■ 8. 재무건전성 평가 기준

【유동성 기준】
- 매우우수: 유동비율 200%↑, 당좌비율 100%↑
- 우수: 유동비율 150-200%, 당좌비율 80-100%
- 보통: 유동비율 100-150%, 당좌비율 50-80%
- 주의: 유동비율 80-100%, 당좌비율 30-50%
- 위험: 유동비율 80%↓, 당좌비율 30%↓

【레버리지 기준】
- 매우우수: 부채비율 100%↓, 이자보상배율 10

In [73]:
핵심 기능 구조
1. 종목 정보 추출 및 매핑
extract_stock_info(): 사용자 질문에서 종목명 추출하고 티커,기업 코드로 변환- 한국/미국 시장 모두 지원

2. 데이터 수집 모듈


get_yfinance_data(): YahooFinance에서 성장성 지표 수집 (시세, 재무제표, 애널리스트 의견, 뉴스)
get_dart_disclosure_data(): DART에서 실적/리스크 지표 수집 (공시, 감사보고서, 재무제표)

3. 분석 엔진

perform_valuation_analysis(): 상대가치법 기반 밸류에이션 (PER, PBR, EV/EBITDA)
assess_investment_risk(): 다차원 리스크 평가 (재무, 감사, 유동성, 지배구조 리스크)

4. 투자 판단 생성

generate_investment_recommendation(): 매수/보류/매도 권고 및 목표가 산출

주요 특징
✅ 통합 데이터 소스: yfinance + DART API 결합으로 성장성과 리스크 동시 평가
✅ 다차원 분석: 정량적 밸류에이션 + 정성적 리스크 평가


SyntaxError: invalid character '✅' (U+2705) (736214709.py, line 21)